# Section 1: Notebook Setup & Imports

This section initializes the environment, resolves the project root, configures reproducibility (seed 42), verifies XGBoost CUDA support, and sets the parallel worker count for multi-model GPU training on H100.

In [ ]:
import os
import sys
import random
import time
import json
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error
import xgboost as xgb
from xgboost import XGBRegressor


def find_project_root():
    candidates = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    for cand in candidates:
        if (cand / "data").exists() and (cand / "d_models").exists():
            return cand
    raise FileNotFoundError("Could not locate repo root containing 'data' and 'd_models'")


PROJECT_ROOT = find_project_root()
print(f"Project root found: {PROJECT_ROOT}")

sys_path_root = str(PROJECT_ROOT)
if sys_path_root not in sys.path:
    sys.path.append(sys_path_root)

out_dir = PROJECT_ROOT / "notebooks/experiment/derived_8.2-hyperparameters-1.4"
out_dir.mkdir(parents=True, exist_ok=True)
print(f"Output directory: {out_dir}")

import warnings
warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)
print(f"Random seed set to {SEED}")

PARALLEL_WORKERS = int(os.environ.get("XGB_PARALLEL_WORKERS", "4"))
print(f"Parallel workers: {PARALLEL_WORKERS} (override with XGB_PARALLEL_WORKERS)")

plt.style.use("default")
plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["axes.grid"] = True

try:
    dummy = xgb.XGBRegressor(n_estimators=1, device="cuda")
    dummy.fit(np.array([[1.0]]), np.array([1.0]))
    XGB_DEVICE = "cuda"
    print("XGBoost CUDA support verified and enabled.")
except Exception as e:
    XGB_DEVICE = "cpu"
    print(f"XGBoost CUDA test failed ({e}). Falling back to CPU.")

print("Setup complete. Using device:", XGB_DEVICE)

# Section 2: Data Loading & Preprocessing

This section loads the Washington-only `derived_8.2` train/validation/test splits, parses dates, extracts month/year, and concatenates train+val into `trainval` for the fixed-budget training protocol used in prior hyperparameter sweeps.

In [ ]:
TRAIN_PATH = PROJECT_ROOT / "data/splits/derived_8.2/train.csv"
VAL_PATH = PROJECT_ROOT / "data/splits/derived_8.2/val.csv"
TEST_PATH = PROJECT_ROOT / "data/splits/derived_8.2/test.csv"

train_df = pd.read_csv(TRAIN_PATH)
val_df = pd.read_csv(VAL_PATH)
test_df = pd.read_csv(TEST_PATH)

print("Dataset splits loaded:")
print(f"  train: {train_df.shape}")
print(f"  val:   {val_df.shape}")
print(f"  test:  {test_df.shape}")

for df in [train_df, val_df, test_df]:
    df["date"] = pd.to_datetime(df["date"])
    df["month"] = df["date"].dt.month.astype(int)
    df["year"] = df["date"].dt.year.astype(float)

trainval_df = pd.concat([train_df, val_df], axis=0).reset_index(drop=True)
print(f"  trainval (concatenated): {trainval_df.shape}")

# Section 3: Feature Set V3 & 200 Hyperparameter Configurations

This section loads Feature Set V3 and programmatically constructs exactly **200** configurations in six groups (R/A/B/C/D/E). Budgets are lean based on `1.3-lite` (val loss flatlines by ~1500 steps). The SOTA control from `1.3-lite` is included as Model 2 so we can measure new SOTA against a fixed baseline.

| Group | Count | Design |
|-------|------:|--------|
| R | 4 | MAE/MSE baselines + SOTA control + Est=2000 |
| A | 72 | depth×MCW×LR around SOTA (d∈{7,8,9}, MCW∈{5,8,10,15}) |
| B | 36 | L2×L1 neighborhood on SOTA backbone |
| C | 16 | subsample×colsample on SOTA backbone |
| D | 48 | lossguide leaves∈{31,63,127,255}×depth×MCW×max_bin |
| E | 24 | gamma, bin×LR, Huber probes, champion combos |


In [ ]:
import importlib.util

metadata_path = PROJECT_ROOT / "data/splits/derived_8.2/dataset_metadata.py"
spec = importlib.util.spec_from_file_location("dataset_metadata", metadata_path)
dataset_metadata = importlib.util.module_from_spec(spec)
spec.loader.exec_module(dataset_metadata)

FEATURE_SET_V3 = dataset_metadata.OVERALL_SELECTED_FEATURES_V3
TARGET_COL = "soil_moisture_5cm"
SOTA_R2_TARGET = 0.655063  # 1.3-lite Model 4 peak

print(f"Feature Set V3 count: {len(FEATURE_SET_V3)}")
print(f"SOTA R2 target to beat: {SOTA_R2_TARGET}")

# Lean n_estimators schedule from 1.3-lite flat-line finding
LEAN_EST = {
    0.005: 2500,
    0.008: 2000,
    0.01: 1500,
    0.012: 1400,
    0.015: 1200,
    0.02: 1000,
    0.04: 1500,
}


def lean_estimators(lr: float) -> int:
    if lr in LEAN_EST:
        return LEAN_EST[lr]
    keys = sorted(LEAN_EST.keys())
    nearest = min(keys, key=lambda k: abs(k - lr))
    return LEAN_EST[nearest]


MODELS_CONFIG = []
group_counts = {}

# ---------------------------------------------------------------------------
# Group R — References (4)
# ---------------------------------------------------------------------------
MODELS_CONFIG.extend([
    {
        "id": 0, "group": "R", "name": "Baseline (MAE)",
        "objective": "reg:absoluteerror", "max_depth": 8, "min_child_weight": 2,
        "reg_lambda": 1.5, "reg_alpha": 0.03, "subsample": 0.9, "colsample_bytree": 0.8,
        "learning_rate": 0.04, "n_estimators": 1500,
    },
    {
        "id": 1, "group": "R", "name": "Baseline (MSE)",
        "objective": "reg:squarederror", "max_depth": 8, "min_child_weight": 2,
        "reg_lambda": 1.5, "reg_alpha": 0.03, "subsample": 0.9, "colsample_bytree": 0.8,
        "learning_rate": 0.04, "n_estimators": 1500,
    },
    {
        "id": 2, "group": "R", "name": "SOTA Control (1.3-lite peak)",
        "objective": "reg:squarederror", "max_depth": 8, "min_child_weight": 10,
        "reg_lambda": 1.5, "reg_alpha": 0.03, "subsample": 0.9, "colsample_bytree": 0.8,
        "learning_rate": 0.01, "n_estimators": 1500,
    },
    {
        "id": 3, "group": "R", "name": "SOTA Control Est=2000",
        "objective": "reg:squarederror", "max_depth": 8, "min_child_weight": 10,
        "reg_lambda": 1.5, "reg_alpha": 0.03, "subsample": 0.9, "colsample_bytree": 0.8,
        "learning_rate": 0.01, "n_estimators": 2000,
    },
])
group_counts["R"] = 4
model_id = 4

# ---------------------------------------------------------------------------
# Group A — Depth x MCW x LR local grid (72)
# ---------------------------------------------------------------------------
for lr in [0.005, 0.008, 0.01, 0.012, 0.015, 0.02]:
    for depth in [7, 8, 9]:
        for mcw in [5, 8, 10, 15]:
            est = lean_estimators(lr)
            MODELS_CONFIG.append({
                "id": model_id,
                "group": "A",
                "name": f"A MSE d={depth} LR={lr} MCW={mcw} Est={est}",
                "objective": "reg:squarederror",
                "max_depth": depth,
                "min_child_weight": mcw,
                "reg_lambda": 1.5,
                "reg_alpha": 0.03,
                "subsample": 0.9,
                "colsample_bytree": 0.8,
                "learning_rate": lr,
                "n_estimators": est,
            })
            model_id += 1
group_counts["A"] = 72

# ---------------------------------------------------------------------------
# Group B — L2 x L1 regularization neighborhood (36)
# ---------------------------------------------------------------------------
for l2 in [0.5, 1.0, 1.5, 2.0, 3.0, 5.0]:
    for l1 in [0.0, 0.01, 0.03, 0.05, 0.1, 0.2]:
        MODELS_CONFIG.append({
            "id": model_id,
            "group": "B",
            "name": f"B MSE L2={l2} L1={l1}",
            "objective": "reg:squarederror",
            "max_depth": 8,
            "min_child_weight": 10,
            "reg_lambda": l2,
            "reg_alpha": l1,
            "subsample": 0.9,
            "colsample_bytree": 0.8,
            "learning_rate": 0.01,
            "n_estimators": 1500,
        })
        model_id += 1
group_counts["B"] = 36

# ---------------------------------------------------------------------------
# Group C — subsample x colsample_bytree (16)
# ---------------------------------------------------------------------------
for sub in [0.8, 0.85, 0.9, 0.95]:
    for col in [0.7, 0.75, 0.8, 0.9]:
        MODELS_CONFIG.append({
            "id": model_id,
            "group": "C",
            "name": f"C MSE sub={sub} col={col}",
            "objective": "reg:squarederror",
            "max_depth": 8,
            "min_child_weight": 10,
            "reg_lambda": 1.5,
            "reg_alpha": 0.03,
            "subsample": sub,
            "colsample_bytree": col,
            "learning_rate": 0.01,
            "n_estimators": 1500,
        })
        model_id += 1
group_counts["C"] = 16

# ---------------------------------------------------------------------------
# Group D — Leaf-wise x max_bin hybrids (48)
# leaves in {31,63,127,255} x depth {6,8} x mcw {8,10,15} x bin {128,256}
# ---------------------------------------------------------------------------
for leaves in [31, 63, 127, 255]:
    for depth in [6, 8]:
        for mcw in [8, 10, 15]:
            for max_bin in [128, 256]:
                MODELS_CONFIG.append({
                    "id": model_id,
                    "group": "D",
                    "name": f"D Leaf={leaves} d={depth} MCW={mcw} Bin={max_bin}",
                    "objective": "reg:squarederror",
                    "grow_policy": "lossguide",
                    "max_leaves": leaves,
                    "max_depth": depth,
                    "min_child_weight": mcw,
                    "max_bin": max_bin,
                    "reg_lambda": 1.5,
                    "reg_alpha": 0.03,
                    "subsample": 0.9,
                    "colsample_bytree": 0.8,
                    "learning_rate": 0.01,
                    "n_estimators": 1500,
                })
                model_id += 1
group_counts["D"] = 48

# ---------------------------------------------------------------------------
# Group E — Gamma / bin x LR / Huber / champion combos (24)
# ---------------------------------------------------------------------------
# E1: gamma on SOTA backbone (6)
for gamma in [0.001, 0.005, 0.01, 0.02, 0.05, 0.1]:
    MODELS_CONFIG.append({
        "id": model_id,
        "group": "E",
        "name": f"E Gamma={gamma}",
        "objective": "reg:squarederror",
        "max_depth": 8,
        "min_child_weight": 10,
        "reg_lambda": 1.5,
        "reg_alpha": 0.03,
        "subsample": 0.9,
        "colsample_bytree": 0.8,
        "learning_rate": 0.01,
        "n_estimators": 1500,
        "gamma": gamma,
    })
    model_id += 1

# E2: max_bin x LR (8)
for max_bin in [64, 128, 256, 512]:
    for lr in [0.01, 0.015]:
        est = lean_estimators(lr)
        MODELS_CONFIG.append({
            "id": model_id,
            "group": "E",
            "name": f"E Bin={max_bin} LR={lr} Est={est}",
            "objective": "reg:squarederror",
            "max_bin": max_bin,
            "max_depth": 8,
            "min_child_weight": 10,
            "reg_lambda": 1.5,
            "reg_alpha": 0.03,
            "subsample": 0.9,
            "colsample_bytree": 0.8,
            "learning_rate": lr,
            "n_estimators": est,
        })
        model_id += 1

# E3: Huber slope 3.0 with SOTA-style MCW (6)
for lr in [0.01, 0.015, 0.02]:
    for l2 in [1.5, 3.0]:
        est = lean_estimators(lr)
        MODELS_CONFIG.append({
            "id": model_id,
            "group": "E",
            "name": f"E Huber3 LR={lr} L2={l2} Est={est}",
            "objective": "reg:pseudohubererror",
            "huber_slope": 3.0,
            "max_depth": 8,
            "min_child_weight": 10,
            "reg_lambda": l2,
            "reg_alpha": 0.03,
            "subsample": 0.9,
            "colsample_bytree": 0.8,
            "learning_rate": lr,
            "n_estimators": est,
        })
        model_id += 1

# E4: champion multi-ingredient combos (4)
champion_specs = [
    {
        "name": "E Champ Leaf127 Bin128 MCW10 LR0.01",
        "objective": "reg:squarederror",
        "grow_policy": "lossguide", "max_leaves": 127, "max_bin": 128,
        "max_depth": 8, "min_child_weight": 10,
        "reg_lambda": 1.5, "reg_alpha": 0.03, "subsample": 0.9, "colsample_bytree": 0.8,
        "learning_rate": 0.01, "n_estimators": 1500,
    },
    {
        "name": "E Champ Leaf127 Bin128 MCW10 LR0.012",
        "objective": "reg:squarederror",
        "grow_policy": "lossguide", "max_leaves": 127, "max_bin": 128,
        "max_depth": 8, "min_child_weight": 10,
        "reg_lambda": 1.5, "reg_alpha": 0.03, "subsample": 0.9, "colsample_bytree": 0.8,
        "learning_rate": 0.012, "n_estimators": 1400,
    },
    {
        "name": "E Champ d=9 Bin128 MCW10 LR0.01",
        "objective": "reg:squarederror",
        "max_bin": 128, "max_depth": 9, "min_child_weight": 10,
        "reg_lambda": 1.5, "reg_alpha": 0.03, "subsample": 0.9, "colsample_bytree": 0.8,
        "learning_rate": 0.01, "n_estimators": 1500,
    },
    {
        "name": "E Champ d=9 MCW12 LR0.01 L2=1.5",
        "objective": "reg:squarederror",
        "max_depth": 9, "min_child_weight": 12,
        "reg_lambda": 1.5, "reg_alpha": 0.03, "subsample": 0.9, "colsample_bytree": 0.8,
        "learning_rate": 0.01, "n_estimators": 1500,
    },
]
for spec in champion_specs:
    cfg = {"id": model_id, "group": "E", **spec}
    MODELS_CONFIG.append(cfg)
    model_id += 1

group_counts["E"] = 24

# ---------------------------------------------------------------------------
# Validation
# ---------------------------------------------------------------------------
ids = [c["id"] for c in MODELS_CONFIG]
assert len(MODELS_CONFIG) == 200, f"Expected 200 configs, got {len(MODELS_CONFIG)}"
assert len(ids) == len(set(ids)), "Duplicate model ids"
assert ids == list(range(200)), "Model ids must be contiguous 0..199"
assert sum(group_counts.values()) == 200

print("Group counts:", group_counts)
print(f"Total model configurations: {len(MODELS_CONFIG)}")
print("SOTA control (id=2):", {k: MODELS_CONFIG[2][k] for k in (
    "objective", "max_depth", "min_child_weight", "reg_lambda", "reg_alpha",
    "subsample", "colsample_bytree", "learning_rate", "n_estimators",
)})


# Section 4: Parallel Training & Timing Loop

This section trains all 200 configurations with a **thread-pool** over the H100 GPU. Models and metadata are cached under `models/` for resume safety. Disk writes use a lock to avoid races. Eval sets record train and test loss at every boosting round for later diagnostics. Set `XGB_PARALLEL_WORKERS=1` if CUDA contention appears.

In [ ]:
y_trainval = np.asarray(trainval_df[TARGET_COL]).ravel()
y_test = np.asarray(test_df[TARGET_COL]).ravel()
X_trainval = trainval_df[FEATURE_SET_V3]
X_test = test_df[FEATURE_SET_V3]

models_dir = out_dir / "models"
models_dir.mkdir(parents=True, exist_ok=True)

test_predictions_df = test_df[["date", "year", "month", TARGET_COL]].copy()
trained_models = {}
evals_results = {}
_save_lock = threading.Lock()
_print_lock = threading.Lock()

OPTIONAL_PARAMS = [
    "grow_policy", "max_leaves", "colsample_bylevel", "colsample_bynode",
    "max_bin", "huber_slope", "gamma",
]


def build_params(config):
    params = {
        "objective": config["objective"],
        "random_state": SEED,
        "n_jobs": 1,  # avoid oversubscription under parallel model training
        "subsample": config["subsample"],
        "colsample_bytree": config["colsample_bytree"],
        "max_depth": config["max_depth"],
        "min_child_weight": config["min_child_weight"],
        "n_estimators": config["n_estimators"],
        "learning_rate": config["learning_rate"],
        "reg_lambda": config["reg_lambda"],
        "reg_alpha": config["reg_alpha"],
        "device": XGB_DEVICE,
    }
    for param_name in OPTIONAL_PARAMS:
        if param_name in config:
            params[param_name] = config[param_name]
    return params


def train_one(config):
    """Train or load a single configuration. Returns (config_id, result_dict)."""
    model_id = config["id"]
    model_name = config["name"]
    model_path = models_dir / f"xgb_model_{model_id}.json"
    meta_path = models_dir / f"xgb_model_{model_id}_meta.json"
    params = build_params(config)

    if model_path.exists() and meta_path.exists():
        with _print_lock:
            print(f"[{model_id}] Loading {model_name} from disk...")
        model = XGBRegressor()
        model.load_model(str(model_path))
        with open(meta_path, "r") as f:
            meta = json.load(f)
        train_time = meta["train_time_s"]
        inference_time = meta["inference_time_s"]
        evals = meta.get("evals_result", {})
        preds = np.asarray(model.predict(X_test)).ravel()
        return model_id, {
            "model": model,
            "preds": preds,
            "train_time_s": train_time,
            "inference_time_s": inference_time,
            "evals": evals,
            "name": model_name,
            "loaded": True,
        }

    with _print_lock:
        print(f"[{model_id}] Training {model_name}...")

    model = XGBRegressor(**params)
    t0 = time.perf_counter()
    model.fit(
        X_trainval, y_trainval,
        eval_set=[(X_trainval, y_trainval), (X_test, y_test)],
        verbose=False,
    )
    t1 = time.perf_counter()
    train_time = t1 - t0

    t2 = time.perf_counter()
    preds = np.asarray(model.predict(X_test)).ravel()
    t3 = time.perf_counter()
    inference_time = t3 - t2

    evals = model.evals_result()
    meta = {
        "train_time_s": train_time,
        "inference_time_s": inference_time,
        "params": {k: v for k, v in params.items() if k != "device"},
        "evals_result": evals,
        "name": model_name,
        "group": config.get("group"),
    }

    # Atomic-ish save under lock (temp + rename)
    with _save_lock:
        tmp_model = model_path.with_suffix(".json.tmp")
        model.save_model(str(tmp_model))
        tmp_model.replace(model_path)
        tmp_meta = meta_path.with_suffix(".json.tmp")
        with open(tmp_meta, "w") as f:
            json.dump(meta, f, indent=2)
        tmp_meta.replace(meta_path)

    with _print_lock:
        print(f"[{model_id}] Done in {train_time:.2f}s (infer {inference_time:.4f}s)")

    return model_id, {
        "model": model,
        "preds": preds,
        "train_time_s": train_time,
        "inference_time_s": inference_time,
        "evals": evals,
        "name": model_name,
        "loaded": False,
    }


print(f"Starting parallel training for {len(MODELS_CONFIG)} configurations "
      f"with {PARALLEL_WORKERS} workers...\n")
wall_t0 = time.perf_counter()
results_by_id = {}
completed = 0

with ThreadPoolExecutor(max_workers=PARALLEL_WORKERS) as executor:
    futures = {executor.submit(train_one, cfg): cfg["id"] for cfg in MODELS_CONFIG}
    for fut in as_completed(futures):
        mid, result = fut.result()
        results_by_id[mid] = result
        completed += 1
        if completed % 20 == 0 or completed == len(MODELS_CONFIG):
            elapsed = time.perf_counter() - wall_t0
            rate = completed / elapsed if elapsed > 0 else 0
            remaining = (len(MODELS_CONFIG) - completed) / rate if rate > 0 else float("nan")
            with _print_lock:
                print(f"  Progress: {completed}/{len(MODELS_CONFIG)} "
                      f"({elapsed:.1f}s elapsed, ~{remaining:.1f}s remaining)")

wall_t1 = time.perf_counter()
print(f"\nAll {len(MODELS_CONFIG)} models processed in {wall_t1 - wall_t0:.1f}s wall time.")

# Assemble predictions and per-config timing in deterministic id order
for config in MODELS_CONFIG:
    mid = config["id"]
    result = results_by_id[mid]
    test_predictions_df[f"pred_{mid}"] = result["preds"]
    config["train_time_s"] = result["train_time_s"]
    config["inference_time_s"] = result["inference_time_s"]
    trained_models[mid] = result["model"]
    evals_results[mid] = result["evals"]

test_predictions_df.to_csv(out_dir / "test_predictions.csv", index=False)
print(f"Saved test predictions to: {out_dir / 'test_predictions.csv'}")

# Step-by-step loss curves
max_steps = 0
for mid in evals_results:
    if not evals_results[mid]:
        continue
    metric_key = list(evals_results[mid]["validation_0"].keys())[0]
    max_steps = max(max_steps, len(evals_results[mid]["validation_0"][metric_key]))

loss_data = {"step": list(range(max_steps))}
for mid in sorted(evals_results.keys()):
    evals = evals_results[mid]
    if not evals:
        loss_data[f"train_loss_model_{mid}"] = [np.nan] * max_steps
        loss_data[f"val_loss_model_{mid}"] = [np.nan] * max_steps
        continue
    metric_key = list(evals["validation_0"].keys())[0]
    train_loss = evals["validation_0"][metric_key]
    val_loss = evals["validation_1"][metric_key]
    loss_data[f"train_loss_model_{mid}"] = list(train_loss) + [np.nan] * (max_steps - len(train_loss))
    loss_data[f"val_loss_model_{mid}"] = list(val_loss) + [np.nan] * (max_steps - len(val_loss))

loss_df = pd.DataFrame(loss_data)
loss_df.to_csv(out_dir / "loss_curves.csv", index=False)
print(f"Saved step-by-step loss history for {len(MODELS_CONFIG)} models to: {out_dir / 'loss_curves.csv'}")

# Section 5: Overall Metrics Summary

This section computes overall test metrics (R², RMSE, ubRMSE, Bias, MAE, median absolute error, Pearson) plus train/inference times for all 200 configurations and saves them to `metrics_summary.csv`.

In [ ]:
def compute_metrics(y_true, y_pred):
    y_true = np.asarray(y_true).ravel()
    y_pred = np.asarray(y_pred).ravel()
    err = y_true - y_pred
    ae = np.abs(err)
    r2 = r2_score(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = root_mean_squared_error(y_true, y_pred)
    ubrmse = np.sqrt(np.mean(((y_true - np.mean(y_true)) - (y_pred - np.mean(y_pred))) ** 2))
    bias = np.mean(err)
    med_ae = np.median(ae)
    if np.std(y_true) == 0 or np.std(y_pred) == 0:
        pearson = float("nan")
    else:
        pearson = np.corrcoef(y_true, y_pred)[0, 1]
    return {
        "R2": r2, "RMSE": rmse, "ubRMSE": ubrmse, "Bias": bias,
        "MAE": mae, "Med|Err|": med_ae, "Pearson": pearson,
    }


summary_records = []
for config in MODELS_CONFIG:
    model_id = config["id"]
    preds = test_predictions_df[f"pred_{model_id}"]
    metrics = compute_metrics(y_test, preds)
    summary_records.append({
        "id": model_id,
        "group": config.get("group", ""),
        "Configuration": config["name"],
        **metrics,
        "Train Time (s)": config["train_time_s"],
        "Inference Time (s)": config["inference_time_s"],
    })

summary_df = pd.DataFrame(summary_records)
cols_order = [
    "id", "group", "Configuration", "R2", "RMSE", "ubRMSE", "Bias", "MAE",
    "Med|Err|", "Pearson", "Train Time (s)", "Inference Time (s)",
]
summary_df = summary_df[cols_order]
summary_df.to_csv(out_dir / "metrics_summary.csv", index=False)

print("===== TOP 20 BY R2 =====")
top20 = summary_df.sort_values("R2", ascending=False).head(20)
print(top20.to_string(index=False, formatters={
    "R2": "{:,.4f}".format,
    "RMSE": "{:,.4f}".format,
    "ubRMSE": "{:,.4f}".format,
    "Bias": "{:+,.4f}".format,
    "MAE": "{:,.4f}".format,
    "Med|Err|": "{:,.4f}".format,
    "Pearson": "{:,.4f}".format,
    "Train Time (s)": "{:,.2f}".format,
    "Inference Time (s)": "{:,.4f}".format,
}))
print(f"\nSaved metrics_summary.csv ({len(summary_df)} rows)")

# Section 6: Year-by-Year Metrics

This section evaluates every configuration on each held-out test year (2023, 2024, 2025) to check temporal generalization, and writes `metrics_by_year.csv`.

In [ ]:
years = sorted(test_df["year"].dropna().unique())
year_records = []
for config in MODELS_CONFIG:
    mid = config["id"]
    preds = test_predictions_df[f"pred_{mid}"].values
    for year in years:
        mask = test_df["year"].values == year
        if mask.sum() == 0:
            continue
        m = compute_metrics(y_test[mask], preds[mask])
        year_records.append({
            "id": mid,
            "group": config.get("group", ""),
            "Configuration": config["name"],
            "year": int(year),
            **m,
        })

metrics_by_year_df = pd.DataFrame(year_records)
metrics_by_year_df.to_csv(out_dir / "metrics_by_year.csv", index=False)
print(f"Saved metrics_by_year.csv ({len(metrics_by_year_df)} rows)")
print("Years:", [int(y) for y in years])
print("\nSOTA control (id=2) by year:")
print(metrics_by_year_df[metrics_by_year_df["id"] == 2][
    ["year", "R2", "RMSE", "MAE"]
].to_string(index=False))

# Section 7: Loss Curves (Selected Models)

Plots train vs test loss for baselines, the SOTA control, and the top-performing configurations after evaluation so we can confirm lean step budgets still flatline.

In [ ]:
# Select models: baselines, SOTA control, group bests, overall top
selected_loss_ids = [0, 1, 2, 3]
best_overall = summary_df.sort_values("R2", ascending=False).head(4)["id"].tolist()
for g in ["A", "B", "C", "D", "E"]:
    gdf = summary_df[summary_df["group"] == g]
    if len(gdf):
        selected_loss_ids.append(int(gdf.sort_values("R2", ascending=False).iloc[0]["id"]))
for mid in best_overall:
    if mid not in selected_loss_ids:
        selected_loss_ids.append(int(mid))
selected_loss_ids = selected_loss_ids[:8]

fig, axes = plt.subplots(4, 2, figsize=(16, 20))
axes = axes.flatten()
for i, mid in enumerate(selected_loss_ids):
    config = next(c for c in MODELS_CONFIG if c["id"] == mid)
    evals = evals_results[mid]
    ax = axes[i]
    if not evals:
        ax.set_title(f"[{mid}] {config['name']} (no evals)")
        continue
    metric_key = list(evals["validation_0"].keys())[0]
    ax.plot(evals["validation_0"][metric_key], label=f"Train ({metric_key})", color="#2a9d8f", linewidth=2)
    ax.plot(evals["validation_1"][metric_key], label=f"Test ({metric_key})", color="#e76f51", linewidth=2)
    r2_val = float(summary_df.loc[summary_df["id"] == mid, "R2"].iloc[0])
    ax.set_xlabel("Boosting Round")
    ax.set_ylabel(metric_key.upper())
    ax.set_title(f"[{mid}] {config['name']}\nR2={r2_val:.4f}")
    ax.grid(True, linestyle="--", alpha=0.6)
    ax.legend()

plt.suptitle("Train vs Test Loss Curves (Selected Configs)", fontsize=16, fontweight="bold", y=0.995)
plt.tight_layout()
plt.savefig(out_dir / "loss_curves.png", dpi=300, bbox_inches="tight")
plt.show()
print(f"Saved loss_curves.png for models: {selected_loss_ids}")

# Section 8: R² by Year Visualization

This section plots overall and year-wise R² for the top configurations and group winners so we can see whether gains hold across 2023–2025 rather than a single year.

In [ ]:
# Top 12 overall + SOTA control + baselines (unique)
plot_ids = [0, 1, 2]
for mid in summary_df.sort_values("R2", ascending=False)["id"].head(12):
    if int(mid) not in plot_ids:
        plot_ids.append(int(mid))
plot_ids = plot_ids[:15]

# Build pivot: config name x year R2 + overall
rows = []
for mid in plot_ids:
    cfg = next(c for c in MODELS_CONFIG if c["id"] == mid)
    overall_r2 = float(summary_df.loc[summary_df["id"] == mid, "R2"].iloc[0])
    row = {"id": mid, "name": f"[{mid}] {cfg['name']}", "Overall": overall_r2}
    ydf = metrics_by_year_df[metrics_by_year_df["id"] == mid]
    for _, r in ydf.iterrows():
        row[str(int(r["year"]))] = r["R2"]
    rows.append(row)

plot_df = pd.DataFrame(rows)
year_cols = [c for c in plot_df.columns if c not in ("id", "name")]
# order: Overall then years
year_order = ["Overall"] + sorted([c for c in year_cols if c != "Overall"])
plot_df = plot_df.set_index("name")[year_order]

fig, ax = plt.subplots(figsize=(14, max(6, 0.45 * len(plot_df))))
x = np.arange(len(plot_df.columns))
width = 0.8 / max(len(plot_df), 1)
for i, (name, row) in enumerate(plot_df.iterrows()):
    ax.bar(x + i * width, row.values, width=width, label=name[:60])
ax.axhline(SOTA_R2_TARGET, color="red", linestyle="--", linewidth=1.5, label=f"SOTA target {SOTA_R2_TARGET:.4f}")
ax.set_xticks(x + width * (len(plot_df) - 1) / 2)
ax.set_xticklabels(plot_df.columns)
ax.set_ylabel("R²")
ax.set_title("R² Overall and by Year — Top Configs vs SOTA Control")
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
ax.grid(True, axis="y", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.savefig(out_dir / "r2_by_year.png", dpi=300, bbox_inches="tight")
plt.show()
print(f"Saved r2_by_year.png for {len(plot_ids)} models")

# Section 9: Residual Plots (Top Configs)

Because 200 residual panels are unreadable, this section plots overall residual scatter for the **top 18** configurations and year-stratified residuals for the **top 6** (including the SOTA control).

In [ ]:
# --- Overall residuals: top 18 ---
top18_ids = summary_df.sort_values("R2", ascending=False).head(18)["id"].astype(int).tolist()
if 2 not in top18_ids:
    top18_ids[-1] = 2  # ensure SOTA control present

fig, axes = plt.subplots(6, 3, figsize=(18, 30))
axes = axes.flatten()
for i, mid in enumerate(top18_ids):
    config = next(c for c in MODELS_CONFIG if c["id"] == mid)
    preds = test_predictions_df[f"pred_{mid}"].values
    res = y_test - preds
    r2_val = float(summary_df.loc[summary_df["id"] == mid, "R2"].iloc[0])
    ax = axes[i]
    ax.scatter(preds, res, s=4, alpha=0.25, c="#3a86c8")
    ax.axhline(0, color="black", linewidth=1)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Residual (true - pred)")
    ax.set_title(f"[{mid}] {config['name'][:50]}\nR2={r2_val:.4f}", fontsize=9)
    ax.grid(True, linestyle="--", alpha=0.4)
for j in range(len(top18_ids), len(axes)):
    axes[j].axis("off")
plt.suptitle("Residuals — Top 18 Configurations", fontsize=16, fontweight="bold", y=0.995)
plt.tight_layout()
plt.savefig(out_dir / "residuals_comparison.png", dpi=200, bbox_inches="tight")
plt.show()
print("Saved residuals_comparison.png")

# --- Residuals by year: top 6 ---
top6_ids = summary_df.sort_values("R2", ascending=False).head(6)["id"].astype(int).tolist()
if 2 not in top6_ids:
    top6_ids = top6_ids[:5] + [2]

unique_years = sorted(int(y) for y in test_df["year"].dropna().unique())
fig, axes = plt.subplots(len(top6_ids), len(unique_years), figsize=(5 * len(unique_years), 3.5 * len(top6_ids)))
if len(top6_ids) == 1:
    axes = np.array([axes])
for i, mid in enumerate(top6_ids):
    config = next(c for c in MODELS_CONFIG if c["id"] == mid)
    preds = test_predictions_df[f"pred_{mid}"].values
    for j, year in enumerate(unique_years):
        ax = axes[i, j]
        mask = test_df["year"].values == year
        res = y_test[mask] - preds[mask]
        r2_y = r2_score(y_test[mask], preds[mask])
        ax.scatter(preds[mask], res, s=4, alpha=0.3, c="#e76f51")
        ax.axhline(0, color="black", linewidth=1)
        ax.set_title(f"[{mid}] {year} R2={r2_y:.4f}", fontsize=9)
        if i == len(top6_ids) - 1:
            ax.set_xlabel("Predicted")
        if j == 0:
            ax.set_ylabel(f"{config['name'][:28]}\nResidual")
        ax.grid(True, linestyle="--", alpha=0.4)
plt.suptitle("Residuals by Year — Top Configs", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(out_dir / "residuals_by_year.png", dpi=200, bbox_inches="tight")
plt.show()
print("Saved residuals_by_year.png")

# Section 10: Leaderboard & New SOTA Check

This section ranks all configurations against the SOTA control (Model 2 / 1.3-lite peak) and flags any **new SOTA** with test R² above 0.655063. Group-wise bests are also summarized for follow-up runs.

In [ ]:
sota_control_r2 = float(summary_df.loc[summary_df["id"] == 2, "R2"].iloc[0])
best_row = summary_df.sort_values("R2", ascending=False).iloc[0]
best_r2 = float(best_row["R2"])
best_id = int(best_row["id"])

print("=" * 72)
print("LEADERBOARD — derived_8.2-hyperparameters-1.4")
print("=" * 72)
print(f"SOTA target (1.3-lite peak):     {SOTA_R2_TARGET:.6f}")
print(f"SOTA control Model 2 (this run): {sota_control_r2:.6f}")
print(f"Best model this sweep:           id={best_id}  R2={best_r2:.6f}")
print(f"  Configuration: {best_row['Configuration']}")
print(f"  Group: {best_row['group']}  Train time: {best_row['Train Time (s)']:.2f}s")

if best_r2 > SOTA_R2_TARGET + 1e-6:
    delta = best_r2 - SOTA_R2_TARGET
    print(f"\n*** NEW SOTA ***  ΔR2 = +{delta:.6f} over 1.3-lite target")
elif best_r2 > sota_control_r2 + 1e-6:
    print(f"\nBest exceeds this-run SOTA control by +{best_r2 - sota_control_r2:.6f} "
          f"(but not the published 1.3-lite target)")
else:
    print("\nNo new SOTA this sweep; SOTA control remains best or tied.")

print("\n----- Top 20 -----")
print(summary_df.sort_values("R2", ascending=False).head(20)[
    ["id", "group", "Configuration", "R2", "RMSE", "MAE", "Train Time (s)"]
].to_string(index=False, formatters={
    "R2": "{:.6f}".format, "RMSE": "{:.4f}".format, "MAE": "{:.4f}".format,
    "Train Time (s)": "{:.2f}".format,
}))

print("\n----- Best per group -----")
for g in ["R", "A", "B", "C", "D", "E"]:
    gdf = summary_df[summary_df["group"] == g]
    if gdf.empty:
        continue
    br = gdf.sort_values("R2", ascending=False).iloc[0]
    print(f"  {g}: id={int(br['id']):3d}  R2={br['R2']:.6f}  {br['Configuration']}")

# Recommend params for the best model
best_cfg = next(c for c in MODELS_CONFIG if c["id"] == best_id)
recommend_keys = [
    "objective", "max_depth", "min_child_weight", "reg_lambda", "reg_alpha",
    "subsample", "colsample_bytree", "n_estimators", "learning_rate",
    "grow_policy", "max_leaves", "max_bin", "gamma", "huber_slope",
]
recommend = {k: best_cfg[k] for k in recommend_keys if k in best_cfg}
print("\n----- Recommended params (best model) -----")
print(json.dumps(recommend, indent=2))

# Persist leaderboard snippet
leaderboard_path = out_dir / "leaderboard_top20.csv"
summary_df.sort_values("R2", ascending=False).head(20).to_csv(leaderboard_path, index=False)
print(f"\nSaved {leaderboard_path}")